# Sensor Data Standardization

## 1. Importing Libraries and Defining Paths

This section imports the required Python libraries and defines
the input and output directories for sensor data standardization.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Find the project root directory
project_path = Path.cwd()

if not (project_path / "environment.yml").exists():
    project_path = project_path.parent

# Input dataset directory
dataset_path = (
    project_path
    / "data"
    / "raw"
    / "TOLIFE_COPD_Dataset"
)

# Output directory
output_path = (
    project_path
    / "data"
    / "interim"
    / "standardized"
    / "sensors"
)

# Create output directory if it doesn't exist
output_path.mkdir(parents=True, exist_ok=True)

print("Project path:", project_path)
print("Dataset exists:", dataset_path.exists())
print("Output path:", output_path)


Project path: C:\Users\Chirag Virdi\Documents\Tech_seva
Dataset exists: True
Output path: C:\Users\Chirag Virdi\Documents\Tech_seva\data\interim\standardized\sensors


## 2. Defining Sensor Streams and Countries

The dataset contains eight different sensor streams
collected from participants in Germany and Spain.

We define these streams and countries so that the same
loading procedure can be applied systematically.

In [2]:

sensor_files = [
    "BRI",
    "EI",
    "MI",
    "PEI",
    "PRVI",
    "SHRI",
    "SII",
    "SQI"
]

countries = [
    "GERMANY",
    "SPAIN"
]

print("Total sensor streams:", len(sensor_files))
print("Countries:", countries)


Total sensor streams: 8
Countries: ['GERMANY', 'SPAIN']


## 3. Loading Sensor Data

This section iterates through all participant folders
in Germany and Spain.

For every participant, it checks the availability of
each sensor CSV file, loads available files, adds
Participant_ID and Country, and normalizes timestamps.

In [3]:

# Create an empty list for every sensor
sensor_data = {
    sensor: []
    for sensor in sensor_files
}

# Track missing sensor files
missing_files = []

# Loop through countries
for country in countries:

    country_path = dataset_path / "index" / country

    if not country_path.exists():
        print("Country folder missing:", country_path)
        continue

    # Loop through participant folders
    for participant_path in sorted(country_path.iterdir()):

        if not participant_path.is_dir():
            continue

        participant_id = participant_path.name

        # Loop through all sensors
        for sensor in sensor_files:

            file_path = participant_path / f"{sensor}.csv"

            # Check whether sensor file exists
            if not file_path.exists():

                missing_files.append({
                    "Participant_ID": participant_id,
                    "Country": country,
                    "Sensor": sensor
                })

                continue

            # Load CSV file
            temp_df = pd.read_csv(file_path)

            # Add participant information
            temp_df["Participant_ID"] = participant_id
            temp_df["Country"] = country

            # Normalize timestamp to UTC
            if "Timestamp" in temp_df.columns:
                temp_df["Timestamp"] = pd.to_datetime(
                    temp_df["Timestamp"],
                    errors="coerce",
                    utc=True
                )

            # Store data according to sensor
            sensor_data[sensor].append(temp_df)

print("Sensor loading completed.")


Sensor loading completed.


## 4. Combining Sensor Data

All participant files belonging to the same sensor
are combined into one DataFrame.

Different sensor schemas are handled using the union
of available columns. Missing values remain missing.

In [4]:

summary = []

for sensor, participant_frames in sensor_data.items():

    if not participant_frames:
        print("No files found for:", sensor)
        continue

    # Combine participant files
    combined_df = pd.concat(
        participant_frames,
        ignore_index=True,
        sort=False
    )

    # Save as Parquet
    output_file = output_path / f"{sensor}.parquet"

    combined_df.to_parquet(
        output_file,
        index=False
    )

    # Collect summary information
    summary.append({
        "Sensor": sensor,
        "Participants_with_files": len(participant_frames),
        "Unique_participants": combined_df["Participant_ID"].nunique(),
        "Total_rows": len(combined_df),
        "Total_columns": len(combined_df.columns)
    })

    print(sensor, "saved successfully.")

summary_df = pd.DataFrame(summary)

display(summary_df)


BRI saved successfully.
EI saved successfully.
MI saved successfully.
PEI saved successfully.
PRVI saved successfully.
SHRI saved successfully.
SII saved successfully.
SQI saved successfully.


,Sensor,Participants_with_files,Unique_participants,Total_rows,Total_columns
0,BRI,74,74,9866,9
1,EI,74,74,19732,21
2,MI,74,74,9908,13
3,PEI,74,74,9907,13
4,PRVI,65,65,2917,17
5,SHRI,74,74,9866,9
6,SII,74,74,19732,9
7,SQI,74,74,9866,11


## 5. Saving Loading Summary

This section saves the participant counts, total rows,
and column counts for each sensor.

It also saves a separate report containing missing
sensor files.

In [5]:

# Save sensor loading summary
summary_df.to_csv(
    output_path / "sensor_load_summary.csv",
    index=False
)

# Save missing sensor file report
missing_files_df = pd.DataFrame(missing_files)

missing_files_df.to_csv(
    output_path / "missing_sensor_files.csv",
    index=False
)

print("Sensor summary saved successfully.")
print("Missing sensor files:", len(missing_files_df))


Sensor summary saved successfully.
Missing sensor files: 9


## 6. Data Validation

This section verifies that all standardized sensor
datasets were saved correctly.

We check the number of records, participants,
and columns in every generated Parquet file.

In [6]:

for sensor in sensor_files:

    file_path = output_path / f"{sensor}.parquet"

    if not file_path.exists():
        print(sensor, "FILE NOT FOUND")
        continue

    df = pd.read_parquet(file_path)

    print(
        sensor,
        "| Rows:", len(df),
        "| Participants:", df["Participant_ID"].nunique(),
        "| Columns:", len(df.columns)
    )


BRI | Rows: 9866 | Participants: 74 | Columns: 9
EI | Rows: 19732 | Participants: 74 | Columns: 21
MI | Rows: 9908 | Participants: 74 | Columns: 13
PEI | Rows: 9907 | Participants: 74 | Columns: 13
PRVI | Rows: 2917 | Participants: 65 | Columns: 17
SHRI | Rows: 9866 | Participants: 74 | Columns: 9
SII | Rows: 19732 | Participants: 74 | Columns: 9
SQI | Rows: 9866 | Participants: 74 | Columns: 11


In [7]:

for sensor in sensor_files:

    file_path = output_path / f"{sensor}.parquet"

    if not file_path.exists():
        continue

    df = pd.read_parquet(file_path)

    print("\nSensor:", sensor)

    print("Participant_ID exists:",
          "Participant_ID" in df.columns)

    print("Country exists:",
          "Country" in df.columns)

    print("data_validity exists:",
          "data_validity" in df.columns)

    print("index_validity exists:",
          "index_validity" in df.columns)

    print("Total columns:", len(df.columns))



Sensor: BRI
Participant_ID exists: True
Country exists: True
data_validity exists: True
index_validity exists: True
Total columns: 9

Sensor: EI
Participant_ID exists: True
Country exists: True
data_validity exists: True
index_validity exists: True
Total columns: 21

Sensor: MI
Participant_ID exists: True
Country exists: True
data_validity exists: True
index_validity exists: True
Total columns: 13

Sensor: PEI
Participant_ID exists: True
Country exists: True
data_validity exists: True
index_validity exists: True
Total columns: 13

Sensor: PRVI
Participant_ID exists: True
Country exists: True
data_validity exists: True
index_validity exists: True
Total columns: 17

Sensor: SHRI
Participant_ID exists: True
Country exists: True
data_validity exists: True
index_validity exists: True
Total columns: 9

Sensor: SII
Participant_ID exists: True
Country exists: True
data_validity exists: True
index_validity exists: True
Total columns: 9

Sensor: SQI
Participant_ID exists: True
Country exists: T

In [8]:
bri_df = pd.read_parquet(
    output_path / "BRI.parquet"
)

display(bri_df.head())

,Timestamp,rf_mean,rf_std,rf_max,rf_min,index_validity,data_validity,Participant_ID,Country
0,2024-05-19 22:00:00+00:00,0.0,0.0,0.0,0.0,0,0,GER-001,GERMANY
1,2024-05-20 22:00:00+00:00,0.0,0.0,0.0,0.0,0,1,GER-001,GERMANY
2,2024-05-21 22:00:00+00:00,0.0,0.0,0.0,0.0,0,1,GER-001,GERMANY
3,2024-05-22 22:00:00+00:00,0.0,0.0,0.0,0.0,0,1,GER-001,GERMANY
4,2024-05-23 22:00:00+00:00,0.0,0.0,0.0,0.0,0,1,GER-001,GERMANY


In [9]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pyarrow

print("PyArrow version:", pyarrow.__version__)

PyArrow version: 26.0.1


In [11]:
import pyarrow
import pandas as pd

print("PyArrow:", pyarrow.__version__)
print("Pandas:", pd.__version__)

PyArrow: 26.0.1
Pandas: 3.1.1


In [12]:
import sys

print("Python executable:")
print(sys.executable)

print("\nTesting PyArrow import:")

try:
    import pyarrow
    print("PyArrow imported successfully!")
    print("Version:", pyarrow.__version__)

except Exception as e:
    print("Import failed!")
    print("Error type:", type(e).__name__)
    print("Error details:", e)

Python executable:
C:\Users\Chirag Virdi\anaconda3\envs\pyarrow_test\python.exe

Testing PyArrow import:
PyArrow imported successfully!
Version: 26.0.1


In [13]:
import sys

print("Python executable:")
print(sys.executable)

Python executable:
C:\Users\Chirag Virdi\anaconda3\envs\pyarrow_test\python.exe


In [14]:
import sys

print("Python executable:")
print(sys.executable)

Python executable:
C:\Users\Chirag Virdi\anaconda3\envs\pyarrow_test\python.exe


In [15]:
import sys
import pandas as pd
import pyarrow

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)

Python: C:\Users\Chirag Virdi\anaconda3\envs\pyarrow_test\python.exe
Pandas: 3.1.1
PyArrow: 26.0.1


In [16]:
from pathlib import Path

print("Output files:")

for file in sorted(output_path.iterdir()):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

print("\nTotal files:", len(list(output_path.iterdir())))

Output files:
BRI.parquet - 28.89 KB
EI.parquet - 178.97 KB
MI.parquet - 93.31 KB
missing_sensor_files.csv - 0.22 KB
PEI.parquet - 96.24 KB
PRVI.parquet - 160.5 KB
sensor_load_summary.csv - 0.22 KB
SHRI.parquet - 29.11 KB
SII.parquet - 72.18 KB
SQI.parquet - 32.53 KB

Total files: 10


In [17]:
import pandas as pd

for sensor in ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]:
    file_path = output_path / f"{sensor}.parquet"

    if file_path.exists():
        df = pd.read_parquet(file_path)
        print(f"{sensor}: {df.shape[0]} rows, {df.shape[1]} columns")
    else:
        print(f"{sensor}: FILE MISSING")

BRI: 9866 rows, 9 columns
EI: 19732 rows, 21 columns
MI: 9908 rows, 13 columns
PEI: 9907 rows, 13 columns
PRVI: 2917 rows, 17 columns
SHRI: 9866 rows, 9 columns
SII: 19732 rows, 9 columns
SQI: 9866 rows, 11 columns


In [18]:
print("SENSOR LOAD SUMMARY")
display(pd.read_csv(output_path / "sensor_load_summary.csv"))

print("MISSING SENSOR FILES")
display(pd.read_csv(output_path / "missing_sensor_files.csv"))

SENSOR LOAD SUMMARY


,Sensor,Participants_with_files,Unique_participants,Total_rows,Total_columns
0,BRI,74,74,9866,9
1,EI,74,74,19732,21
2,MI,74,74,9908,13
3,PEI,74,74,9907,13
4,PRVI,65,65,2917,17
5,SHRI,74,74,9866,9
6,SII,74,74,19732,9
7,SQI,74,74,9866,11


MISSING SENSOR FILES


,Participant_ID,Country,Sensor
0,GER-020,GERMANY,PRVI
1,GER-026,GERMANY,PRVI
2,GER-030,GERMANY,PRVI
3,GER-033,GERMANY,PRVI
4,GER-037,GERMANY,PRVI
5,GER-039,GERMANY,PRVI
6,SPA-013,SPAIN,PRVI
7,SPA-040,SPAIN,PRVI
8,SPA-046,SPAIN,PRVI
